# ETL from CSV to Oracle using pandas and oracledb

This notebook demonstrates a complete ETL pipeline: extracting data from CSV files, transforming and cleaning the data, saving the result, and loading it into an Oracle database. Progress is logged throughout the process.

## 1. Import Required Libraries
Import all necessary libraries: oracledb, glob, pandas, and datetime.

In [ ]:
import oracledb
import glob
import pandas as pd
from datetime import datetime

## 2. Configure Oracle Database Connection
Set up Oracle database credentials and DSN as variables.

In [ ]:
# Oracle DB credentials
DB_USER = "HR"
DB_PASSWORD = "hr"
DB_DSN = "localhost:1521/xe"

# Target CSV file
log_file = "log_file.txt"
target_file = "transformed_data.csv"

# Expected schema for ETL + Oracle insert
EXPECTED_COLS = [
    "partnumber", "location", "type", "quantity", "unit",
    "expdate", "parenttype", "classes", "segment", "lotcode",
    "status", "value", "currency", "source", "storedate"
]

## 3. Extract Data from CSV Files
Use glob and pandas to read all CSV files in the directory (excluding the target file), normalize column names, and concatenate them into a single DataFrame.

In [ ]:
def extract_from_csv(file_to_process):
    df = pd.read_csv(file_to_process)
    # Normalize column names
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace('\ufeff', '', regex=True)
    )
    # Drop unnamed and duplicate columns
    df = df.loc[:, ~df.columns.str.contains('^unnamed')]
    df = df.loc[:, ~df.columns.duplicated()]
    return df

# Extract all CSVs except the target file
extracted_data = pd.DataFrame()
for csvfile in glob.glob("*.csv"):
    if csvfile != target_file:
        df = extract_from_csv(csvfile)
        extracted_data = pd.concat([extracted_data, df], ignore_index=True)

extracted_data.head()

## 4. Transform Extracted Data
Clean the DataFrame: remove unnamed/duplicate columns, fix column typos, add missing columns, drop unexpected columns, and enforce column order.

In [ ]:
def transform(df):
    # Remove unnamed and duplicate columns
    df = df.loc[:, ~df.columns.str.contains('^unnamed')]
    df = df.loc[:, ~df.columns.duplicated()]
    # Fix typos
    df = df.rename(columns={
        "parentyype": "parenttype",
        "class": "classes"
    })
    # Add missing expected columns
    for col in EXPECTED_COLS:
        if col not in df.columns:
            df[col] = None
    # Drop unexpected columns and enforce correct order
    df = df[EXPECTED_COLS]
    return df

transformed_data = transform(extracted_data)
transformed_data.head()

## 5. Save Transformed Data to CSV
Save the cleaned DataFrame to a CSV file using pandas.

In [ ]:
transformed_data.to_csv(target_file, index=False)
print(f"Transformed data saved to {target_file}")

## 6. Insert Data into Oracle Database
Convert NaN to None, prepare data as tuples, and use oracledb to insert the data into the Oracle table with executemany.

In [ ]:
# Convert NaN → None for Oracle
oracle_data = transformed_data.astype(object).where(pd.notna(transformed_data), None)
data_to_insert = [tuple(row) for row in oracle_data.values]

sql = """
INSERT INTO HR.INVENTORY_LOT  (
    partnumber, location, type, quantity, unit,
    expdate, parenttype, classes, segment, lotcode,
    status, value, currency, source, storedate
) VALUES (
    :1, :2, :3, :4, :5,
    :6, :7, :8, :9, :10,
    :11, :12, :13, :14, :15
)
"""

try:
    with oracledb.connect(user=DB_USER, password=DB_PASSWORD, dsn=DB_DSN) as connection:
        print("Connected to Oracle Database")
        with connection.cursor() as cursor:
            cursor.executemany(sql, data_to_insert)
            connection.commit()
            print("Data successfully inserted into inventorylot")
except oracledb.Error as error:
    print(f"Error connecting to ORACLE DB: {error}")

## 7. Log ETL Progress
Log ETL job progress with timestamps to a log file using Python's datetime and file I/O.

In [ ]:
def log_progress(message, log_file=log_file):
    timestamp = datetime.now().strftime('%Y-%b-%d-%H:%M:%S')
    with open(log_file, "a") as f:
        f.write(f"{timestamp}, {message}\n")

# Example usage:
log_progress("ETL Job Started")
log_progress("Extract phase Started")
log_progress("Extract phase Ended")
log_progress("Transform phase Started")
log_progress("Transform phase Ended")
log_progress("Load phase Started")
log_progress("Load phase Ended")
log_progress("ETL Job Ended")